In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install python-mecab-ko

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.9/573.9 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 31.7 MB/s eta 0:00:00


In [3]:
import re
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from mecab import MeCab
mecab = MeCab()
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = 75
import pickle

In [4]:
# 저장된 모델 로드
re_bst_loaded = lgb.Booster(model_file='/content/drive/MyDrive/2023데청캠/팀플/합불예측/LGBMmodel.txt')

#Tokenizer Object 파일로드
with open('/content/drive/MyDrive/2023데청캠/팀플/합불예측/tokenizer.pickle', 'rb') as handle:
    loaded_tokenizer = pickle.load(handle)

In [5]:
def predict_sen(new_sentence, model):
    new_sentence = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣 ]','', new_sentence)
    new_sentence = mecab.morphs(new_sentence) # 토큰화
    encoded = loaded_tokenizer.texts_to_sequences([new_sentence]) # 정수 인코딩
    pad_new = pad_sequences(encoded, maxlen = max_len)

    pred_prob = float(model.predict(pad_new))
    print(pred_prob)

    # 결과 출력
    if pred_prob >= 0.5:
        result = f"This sentence is {pred_prob*100:.2f}% likely to be 합격."
    else:
        result = f"This sentence is {100-pred_prob*100:.2f}% likely to be 불합격."

    return result

In [6]:
#사람인 면접후기(합격)
input_sentence = """저는 긴장 많이 했습니다. 그래서 그런지 쉽다고 생각했지만, 면접관분들이 그런 부분을 이해하고 많이 풀어주시려 노력합니다. 정확하지 않다고 무작정 모른다는 자세보다 아는 만큼 설명하고 대답에 자신감을 가지면서 답변드리면 좋은 결과가 있을 거라고 생각됩니다. 화이팅입니다."""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.19812648496683447
This sentence is 80.19% likely to be 불합격.


In [7]:
#사람인 면접후기(합격)
input_sentence = """다른 호텔에 비해 굉장히 간단한것만 물어보셨고, 지원자 한 명, 면접관 두 명이라 마음 편히 볼 수 있었음"""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.8848346575073908
This sentence is 88.48% likely to be 합격.


In [8]:
#사람인 면접후기(합격)
input_sentence = """분위기는 굉장히 편하게 해주십니다. 대기시간이 길기 때문에 대기시간에 뭘 할지 생각해 두시면 좋습니다. 저는 전공 필기내용을 다시 읽어서 그에 대한 질답을 준비하고 직무에 대한 내용을 휴대폰으로 한번 더 찾아봤습니다"""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.3996242579674478
This sentence is 60.04% likely to be 불합격.
